# ME4.2: Deutsch-Jozsa (2-qubit)

## Objectives
- Implement 2-qubit DJ; classify $f$ as CONSTANT vs BALANCED in one shot.
- Show oracle families (constant 0/1, balanced permutations).

## Setup
```python
from qiskit import QuantumCircuit
from qiskit_aer import Aer
```


## Theory Snapshot
- $H^{\otimes n}-oracle-H^{\otimes n}$ maps constant $f\rightarrow∣00\rangle$, balanced $f\rightarrow$ non-zero outputs.
- Deterministic in ideal/noiseless simulation.

## Experiment


In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import Aer

sim = Aer.get_backend('aer_simulator')

# --- Oracle builders (n=2 inputs, ancilla at qubit 2) ---
def oracle_const0(qc):  # f(x)=0
    pass

def oracle_const1(qc):  # f(x)=1
    qc.x(2)

def oracle_bal_x0(qc):  # f(x)=x0
    qc.cx(0, 2)

def oracle_bal_x1(qc):  # f(x)=x1
    qc.cx(1, 2)

def oracle_bal_x0_x1(qc):  # f(x)=x0 XOR x1
    qc.cx(0, 2); qc.cx(1, 2)

def build_dj_circuit(oracle_builder):
    qc = QuantumCircuit(3, 2)     # 2 inputs (0,1), 1 ancilla (2)
    qc.x(2); qc.h([0,1,2])        # |1> ancilla, H on all
    oracle_builder(qc)            
    qc.h([0,1])                   # H on inputs
    qc.measure([0,1],[0,1])       # measure inputs only
    return qc

def run_and_classify(name, oracle):
    qc = build_dj_circuit(oracle)
    res = sim.run(qc, shots=1024).result().get_counts()
    verdict = "CONSTANT" if res.get('00', 0) == 1024 else "BALANCED"
    print(f"{name:>16}: counts={res} , verdict: {verdict}")

cases = [
    ("const 0", oracle_const0),
    ("const 1", oracle_const1),
    ("bal x0",  oracle_bal_x0),
    ("bal x1",  oracle_bal_x1),
    ("bal x0^x1", oracle_bal_x0_x1),
]

for name, oracle in cases:
    run_and_classify(name, oracle)


## Results & Discussion

### Results
- Prints: e.g., const 0 -> {'00':1024}, bal x0^x1 -> {'11':1024} with correct verdicts.
- One-shot correctness achieved for tested oracles.

### Discussion
- Matches textbook DJ behavior under ideal Aer.
- Real devices would require repetitions due to noise/measurement errors.